In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import os
from text_utils import clean_city


CITIES = ["Toronto", "Montreal"]

In [2]:
df_business = pd.read_csv("../data/raw/yelp_business.csv")
df_business.head(3)

,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories
0,FYWN1wneV18bWNgQjJ2GNg,"""Dental by Design""",NaN,"""4855 E Warner Rd, Ste B9""",Ahwatukee,AZ,85044,33.330690,-111.978599,4.0,22,1,Dentists;General Dentistry;Health & Medical;Or...
1,He-G7vWjzVUysIKrfNbPUQ,"""Stephen Szabo Salon""",NaN,"""3101 Washington Rd""",McMurray,PA,15317,40.291685,-80.104900,3.0,11,1,Hair Stylists;Hair Salons;Men's Hair Salons;Bl...
2,KQPW8lFf1y5BT2MxiSZ3QA,"""Western Motor Vehicle""",NaN,"""6025 N 27th Ave, Ste 1""",Phoenix,AZ,85017,33.524903,-112.115310,1.5,18,1,Departments of Motor Vehicles;Public Services ...


In [3]:
df_business["city"].value_counts()[:15]

city
Las Vegas     26775
Phoenix       17213
Toronto       17206
Charlotte      8553
Scottsdale     8228
Pittsburgh     6355
Mesa           5760
Montréal       5709
Henderson      4465
Tempe          4263
Chandler       3994
Edinburgh      3868
Cleveland      3322
Madison        3213
Glendale       3206
Name: count, dtype: int64

In [4]:
[v for v in list(df_business["city"].unique()) if "tor" in str(v).lower()]

['Toronto',
 'Mentor',
 'Mentor-on-the-Lake',
 'Mentor On the Lake',
 'toronto',
 'Mentor On the',
 'Toronto, Ontario',
 'Mentor On The Lake',
 'West Toronto',
 'Downtown Toronto',
 'North Toronto',
 'Tornto',
 'Toronto Division']

In [5]:
# Apply to the dataframe
df_business["city_clean"] = df_business["city"].apply(clean_city)

In [6]:
df_business["city_clean"].value_counts()[:15]

city_clean
Las Vegas     26775
Toronto       17221
Phoenix       17213
Charlotte      8553
Scottsdale     8228
Pittsburgh     6355
Montreal       6056
Mesa           5760
Henderson      4465
Tempe          4263
Chandler       3994
Edinburgh      3868
Cleveland      3322
Madison        3213
Glendale       3206
Name: count, dtype: int64

In [7]:
[v for v in list(df_business["city_clean"].unique()) if "tor" in str(v).lower() or "nto" in str(v).lower()]

['Toronto',
 'Mentor',
 'Mentor-on-the-Lake',
 'Rantoul',
 'Downtown',
 'Mentor On the Lake',
 'Mentor On the',
 'Mentor On The Lake',
 'clinton',
 'Clinton']

In [8]:
[v for v in list(df_business["city_clean"].unique()) if "mont" in str(v).lower()]

['Montreal',
 'Belmont',
 'Monticello',
 'Oakmont',
 'Tremont',
 'Dormont',
 'Saint-Bruno-de-Montarville',
 'Mont-Saint-Gregoire',
 'Mont-Saint-Hilaire',
 'Vimont',
 'Piedmont',
 'Montrose',
 'Deux-Montagnes Regional County Municipality',
 'Saint-Sauveur-des-Monts',
 'St-Bruno-de-Montarville',
 'Mont St-hilaire',
 'Claremont']

In [9]:
df_business = df_business.loc[df_business["city_clean"].isin(CITIES)].copy()
len(df_business)

23277

In [11]:
df_checkin = pd.read_csv("../data/raw/yelp_checkin.csv")
print(df_checkin.shape)
df_checkin.head(3)

(3911218, 4)


,business_id,weekday,hour,checkins
0,3Mc-LxcqeguOXOVT_2ZtCg,Tue,0:00,12
1,SVFx6_epO22bZTZnKwlX7g,Wed,0:00,4
2,vW9aLivd4-IorAfStzsHww,Tue,14:00,1


In [12]:
df_checkin.info()

<class 'pandas.DataFrame'>
RangeIndex: 3911218 entries, 0 to 3911217
Data columns (total 4 columns):
 #   Column       Dtype
---  ------       -----
 0   business_id  str  
 1   weekday      str  
 2   hour         str  
 3   checkins     int64
dtypes: int64(1), str(3)
memory usage: 119.4 MB


In [13]:
# Step 1: Find the peak hour (max checkins) per business + weekday
peak_hours = (
    df_checkin
    .groupby(["business_id", "weekday", "hour"], as_index=False)
    .agg(avg_checkins=("checkins", "mean"))
    .sort_values("avg_checkins", ascending=False)
    .groupby(["business_id", "weekday"], as_index=False)
    .first()  # now "first" = the hour with highest *average* checkins
    .rename(columns={"hour": "peak_hour", "avg_checkins": "peak_avg_checkins"})
)

# Step 2: Also get total checkins per business + weekday (useful context)
totals = (
    df_checkin
    .groupby(["business_id", "weekday"], as_index=False)
    .agg(total_checkins=("checkins", "sum"))
)

# Step 3: Merge both
df_peak = peak_hours.merge(totals, on=["business_id", "weekday"])

# Optional: sort weekdays in calendar order
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df_peak["weekday"] = pd.Categorical(df_peak["weekday"], categories=weekday_order, ordered=True)
df_peak = df_peak.sort_values(["business_id", "weekday"])

df_peak.head(14)

,business_id,weekday,peak_hour,peak_avg_checkins,total_checkins
1,--6MefnULPED_I942VcFNA,Mon,0:00,6.0,19
5,--6MefnULPED_I942VcFNA,Tue,23:00,3.0,7
6,--6MefnULPED_I942VcFNA,Wed,22:00,3.0,8
4,--6MefnULPED_I942VcFNA,Thu,23:00,2.0,7
0,--6MefnULPED_I942VcFNA,Fri,23:00,5.0,21
2,--6MefnULPED_I942VcFNA,Sat,23:00,12.0,40
3,--6MefnULPED_I942VcFNA,Sun,1:00,6.0,37
8,--7zmmkVg-IMGaXbuVd0SQ,Mon,21:00,2.0,6
12,--7zmmkVg-IMGaXbuVd0SQ,Tue,22:00,6.0,7
13,--7zmmkVg-IMGaXbuVd0SQ,Wed,20:00,2.0,6


In [14]:
len(df_peak)

733576

In [15]:
print(df_business.shape)
df = df_business.merge(df_peak, on="business_id", how= "left")
print(df.shape)

df.head(3)

(23277, 14)
(104665, 18)


,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories,city_clean,weekday,peak_hour,peak_avg_checkins,total_checkins
0,l09JfMeQ6ynYs5MCJtrcmQ,"""Alize Catering""",Yonge and Eglinton,"""2459 Yonge St""",Toronto,ON,M4P 2H6,43.711399,-79.399339,3.0,12,0,Italian;French;Restaurants,Toronto,Wed,22:00,1.0,1.0
1,l09JfMeQ6ynYs5MCJtrcmQ,"""Alize Catering""",Yonge and Eglinton,"""2459 Yonge St""",Toronto,ON,M4P 2H6,43.711399,-79.399339,3.0,12,0,Italian;French;Restaurants,Toronto,Thu,3:00,1.0,1.0
2,l09JfMeQ6ynYs5MCJtrcmQ,"""Alize Catering""",Yonge and Eglinton,"""2459 Yonge St""",Toronto,ON,M4P 2H6,43.711399,-79.399339,3.0,12,0,Italian;French;Restaurants,Toronto,Sat,22:00,1.0,1.0


In [16]:
df.to_csv("../data/processed/yelp_business_data_cleaned.csv", index=False)

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104665 entries, 0 to 104664
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   business_id        104665 non-null  str     
 1   name               104665 non-null  str     
 2   neighborhood       91745 non-null   str     
 3   address            104665 non-null  str     
 4   city               104665 non-null  str     
 5   state              104665 non-null  str     
 6   postal_code        104243 non-null  str     
 7   latitude           104660 non-null  float64 
 8   longitude          104660 non-null  float64 
 9   stars              104665 non-null  float64 
 10  review_count       104665 non-null  int64   
 11  is_open            104665 non-null  int64   
 12  categories         104665 non-null  str     
 13  city_clean         104665 non-null  str     
 14  weekday            101885 non-null  category
 15  peak_hour          101885 non-null  str     
